# 04 — Colab Training (T4 GPU)
Run this notebook **top-to-bottom** on a Colab T4 runtime.  
Go to **Runtime → Change runtime type → T4 GPU** before starting.

In [ ]:
# Cell 1 — Verify GPU
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU found — switch to T4 runtime (Runtime > Change runtime type)"
print(f"CUDA: {torch.version.cuda} | Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2 — Install dependencies
!pip install -q ultralytics albumentations scikit-learn

In [ ]:
# Cell 3 — Clone repo
import os
REPO_URL = "https://github.com/EdwinRivera04/wildfire-smoke-detection.git"
REPO_DIR = "/content/wildfire-smoke-detection"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
print("Working dir:", os.getcwd())

In [ ]:
# Cell 4 — Download D-Fire dataset from Kaggle
# Option A: Kaggle CLI (requires kaggle.json)
# Upload your kaggle.json first: Files panel → Upload
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d sayedgamal99/smoke-fire-detection-yolo -p data/
!unzip -q data/smoke-fire-detection-yolo.zip -d data/raw/
!rm data/smoke-fire-detection-yolo.zip
print("Dataset downloaded.")

In [ ]:
# Cell 5 — Run prepare_dataset.py to build data/processed/
!python src/data/prepare_dataset.py

In [ ]:
# Cell 6 — Patch configs for CUDA (T4)
import yaml, copy

CUDA_OVERRIDES = {"device": "cuda", "workers": 4, "imgsz": 640, "batch": 16}
MPS_RESTORE    = {"device": "mps",  "workers": 0, "imgsz": 416, "batch": 32}

def patch_config(path, overrides):
    with open(path) as f:
        cfg = yaml.safe_load(f)
    cfg.update(overrides)
    with open(path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)

patch_config("configs/baseline.yaml", CUDA_OVERRIDES)
patch_config("configs/improved.yaml", CUDA_OVERRIDES)
print("Configs patched for CUDA:")
!cat configs/baseline.yaml | grep -E 'device|workers|imgsz|batch'

In [ ]:
# Cell 7 — Train baseline (25 epochs, ~17 min on T4)
import time
t0 = time.time()
!python src/train.py --config configs/baseline.yaml
baseline_time = time.time() - t0
print(f"\nBaseline training: {baseline_time/60:.1f} min")

In [ ]:
# Cell 8 — Train improved (15 epochs, ~11 min on T4)
t0 = time.time()
!python src/train.py --config configs/improved.yaml
improved_time = time.time() - t0
print(f"\nImproved training: {improved_time/60:.1f} min")
print(f"Total: {(baseline_time + improved_time)/60:.1f} min")

In [ ]:
# Cell 9 — Mount Google Drive and save weights
from google.colab import drive
drive.mount("/content/drive")

import shutil
from pathlib import Path

drive_dir = Path("/content/drive/MyDrive/wildfire-weights")
drive_dir.mkdir(parents=True, exist_ok=True)

baseline_pt = Path("outputs/checkpoints/baseline/weights/best.pt")
improved_pt = Path("outputs/checkpoints/improved/weights/best.pt")

assert baseline_pt.exists(), f"Missing: {baseline_pt}"
assert improved_pt.exists(), f"Missing: {improved_pt}"

shutil.copy(baseline_pt, drive_dir / "baseline_best.pt")
shutil.copy(improved_pt, drive_dir / "improved_best.pt")
print(f"Weights saved to: {drive_dir}")
print("Share this Drive folder link with your teammates.")

In [ ]:
# Cell 10 — Restore configs to MPS values and push
patch_config("configs/baseline.yaml", MPS_RESTORE)
patch_config("configs/improved.yaml", {**MPS_RESTORE, "batch": 32})
print("Configs restored to MPS values.")

# Commit restored configs (optional — set your Git credentials first)
# !git config user.email "your@email.com"
# !git config user.name "Lyla"
# !git add configs/
# !git commit -m "restore configs to MPS values after Colab training"
# !git push

In [ ]:
# Cell 11 — Summary
print("=" * 60)
print("TRAINING COMPLETE")
print(f"  Baseline : {baseline_time/60:.1f} min")
print(f"  Improved : {improved_time/60:.1f} min")
print(f"  Total    : {(baseline_time+improved_time)/60:.1f} min")
print()
print(f"Weights saved to Google Drive: {drive_dir}")
print()
print("Next steps for teammates:")
print("  1. Lyla shares the Drive folder link")
print("  2. Edwin & Ann download baseline_best.pt and improved_best.pt")
print("  3. Place at outputs/checkpoints/{baseline,improved}/weights/best.pt")
print("  4. Ann runs: python src/evaluate.py")
print("  5. Ann runs: python src/demo.py")
print("=" * 60)